# TB1 – Dataset y análisis exploratorio
**Curso:** 1FIS0281 Analítica de Datos y Sistemas Predictivos · **Caso:** IE 20395 – rendimiento académico
**Responsable:** Omar (Dataset y análisis exploratorio)

**Cómo usar este cuaderno cuando llegue la data real:**
1. Copia el archivo (CSV o Excel) a la misma carpeta del cuaderno.
2. Edita **solo la celda 1 (CONFIG)**: nombre del archivo, nombres de columnas objetivo, identificadores y columnas sensibles.
3. Ejecuta todo (`Run All`). Si el archivo no existe, el cuaderno corre con **datos sintéticos de demostración** (se avisa en pantalla). Esos resultados NO sirven para el informe.
4. Entrega a tus compañeros los archivos de la carpeta `salida/`: dataset limpio y anonimizado, diccionario de datos y particiones train/test.

Cada sección dice qué parte del Word (3.1) alimenta.

## 1. Configuración (lo único que debes editar)

In [ ]:
CONFIG = {
    # --- Archivo de datos ---
    "DATA_PATH": "datos_ie20395.csv",   # .csv, .xlsx o .xls
    "SEP": ",",                          # solo CSV (si sale en una sola columna, prueba ";")
    "ENCODING": "utf-8",                 # si hay tildes rotas, prueba "latin-1"
    "SHEET": 0,                          # solo Excel: nombre o número de hoja

    # --- Variables objetivo ---
    "TARGET_REG": "nota_final",          # regresión: nota final numérica
    "PASS_THRESHOLD": 11,                # nota mínima aprobatoria (escala vigesimal: 11)
    "SCALE_MIN": 0,
    "SCALE_MAX": 20,
    "TARGET_CLF": "riesgo_academico",    # se crea: 1 = en riesgo (nota < umbral), 0 = no

    # --- Privacidad (datos de menores) ---
    "ID_COLS": ["codigo_estudiante", "dni"],        # se reemplazan por un seudónimo irreversible
    "NAME_COLS": ["nombres", "apellidos"],          # se eliminan
    "OTHER_SENSITIVE_COLS": [],                     # direcciones, teléfonos, correos, etc. (se eliminan)

    # --- Fuga de información (columnas calculadas a partir de la nota final) ---
    "LEAKAGE_COLS": [],                  # ej. "promedio_anual", "condicion_final"
    "DROP_COLS": [],                     # cualquier otra columna que no se use

    # --- Reproducibilidad ---
    "SEED": 42,
    "TEST_SIZE": 0.2,
    "OUTPUT_DIR": "salida",
}

## 2. Librerías y carga de datos

In [ ]:
import os, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

C = CONFIG
os.makedirs(C["OUTPUT_DIR"], exist_ok=True)
rng = np.random.default_rng(C["SEED"])

In [ ]:
def generar_datos_demo(n=600, seed=0):
    """Datos SINTÉTICOS solo para probar el cuaderno. No representan a la IE 20395."""
    r = np.random.default_rng(seed)
    grado = r.choice(["1ro", "2do", "3ro", "4to", "5to"], n)
    asistencia = np.clip(r.normal(90, 8, n), 40, 100).round(1)
    horas = np.clip(r.normal(8, 4, n), 0, 25).round(1)
    base = 6 + 0.07 * asistencia + 0.25 * horas + r.normal(0, 2, n)
    b1 = np.clip(base + r.normal(0, 1.5, n), 0, 20).round()
    b2 = np.clip(base + r.normal(0, 1.5, n), 0, 20).round()
    b3 = np.clip(base + r.normal(0, 1.5, n), 0, 20).round()
    final = np.clip(((b1 + b2 + b3) / 3 + r.normal(0, 1, n)), 0, 20).round()
    df = pd.DataFrame({
        "codigo_estudiante": [f"E{100000 + i}" for i in range(n)],
        "dni": r.integers(70000000, 79999999, n).astype(str),
        "nombres": ["Nombre" + str(i) for i in range(n)],
        "apellidos": ["Apellido" + str(i) for i in range(n)],
        "grado": grado,
        "seccion": r.choice(["A", "B", "C"], n),
        "sexo": r.choice(["F", "M"], n),
        "edad": r.integers(11, 18, n),
        "asistencia_pct": asistencia,
        "tardanzas": r.poisson(3, n),
        "horas_estudio_sem": horas,
        "trabaja": r.choice(["Si", "No"], n, p=[0.15, 0.85]),
        "vive_con_padres": r.choice(["Si", "No"], n, p=[0.8, 0.2]),
        "nota_bim1": b1, "nota_bim2": b2, "nota_bim3": b3,
        "nota_final": final,
    })
    # Defectos intencionales para probar la limpieza
    for col, frac in [("asistencia_pct", 0.04), ("horas_estudio_sem", 0.06), ("nota_bim2", 0.03)]:
        df.loc[r.choice(n, int(n * frac), replace=False), col] = np.nan
    df.loc[r.choice(n, 5, replace=False), "nota_bim1"] = 25      # fuera de escala
    df = pd.concat([df, df.sample(8, random_state=1)], ignore_index=True)  # duplicados
    return df


def cargar_datos(cfg):
    ruta = cfg["DATA_PATH"]
    if not os.path.exists(ruta):
        print(f"AVISO: no se encontró '{ruta}'. Usando DATOS SINTÉTICOS DE DEMOSTRACIÓN.")
        print("       Los resultados de esta ejecución NO deben usarse en el informe.")
        return generar_datos_demo(), True
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta, sheet_name=cfg["SHEET"]), False
    return pd.read_csv(ruta, sep=cfg["SEP"], encoding=cfg["ENCODING"]), False


df_raw, ES_DEMO = cargar_datos(C)
df = df_raw.copy()
print(f"Registros: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

## 3. Resumen del dataset y diccionario de datos
*Alimenta: 3.1 «Resumen del dataset (columnas, descripción, cantidad de registros)».*

In [ ]:
# Descripciones: completa este diccionario con el significado real de cada columna.
# Las que no estén aquí quedan como "(completar)".
DESCRIPCIONES = {
    "grado": "Grado de estudios del estudiante",
    "seccion": "Sección del aula",
    "sexo": "Sexo del estudiante",
    "edad": "Edad en años",
    "asistencia_pct": "Porcentaje de asistencia en el periodo",
    "tardanzas": "Cantidad de tardanzas",
    "horas_estudio_sem": "Horas de estudio por semana",
    "trabaja": "Si el estudiante trabaja (Si/No)",
    "vive_con_padres": "Si vive con sus padres (Si/No)",
    "nota_bim1": "Nota del bimestre 1 (0-20)",
    "nota_bim2": "Nota del bimestre 2 (0-20)",
    "nota_bim3": "Nota del bimestre 3 (0-20)",
    "nota_final": "Nota final del año (0-20)",
}


def tipo_legible(s):
    if pd.api.types.is_numeric_dtype(s):
        return "Numérica"
    return "Categórica / texto"


def diccionario(d):
    filas = []
    for col in d.columns:
        s = d[col]
        ej = s.dropna().astype(str).unique()[:3]
        filas.append({
            "columna": col,
            "tipo": tipo_legible(s),
            "dtype": str(s.dtype),
            "no_nulos": int(s.notna().sum()),
            "nulos": int(s.isna().sum()),
            "% nulos": round(s.isna().mean() * 100, 2),
            "valores_unicos": int(s.nunique()),
            "ejemplos": ", ".join(ej),
            "descripcion": DESCRIPCIONES.get(col, "(completar)"),
        })
    return pd.DataFrame(filas)


dic_raw = diccionario(df_raw)
print(f"El dataset original tiene {len(df_raw):,} registros y {df_raw.shape[1]} columnas.")
dic_raw

## 4. Calidad de datos: faltantes, duplicados, valores fuera de rango

In [ ]:
# 4.1 Faltantes
faltantes = df.isna().sum().to_frame("nulos")
faltantes["%"] = (faltantes["nulos"] / len(df) * 100).round(2)
faltantes = faltantes[faltantes["nulos"] > 0].sort_values("nulos", ascending=False)
print("Columnas con datos faltantes:")
display(faltantes) if len(faltantes) else print("  Ninguna")

# 4.2 Duplicados (se excluyen las columnas identificadoras para detectar filas repetidas por error)
n_dup_exactos = int(df.duplicated().sum())
print(f"\nFilas duplicadas exactas: {n_dup_exactos}")
id_presentes = [c for c in C["ID_COLS"] if c in df.columns]
if id_presentes:
    n_dup_id = int(df.duplicated(subset=id_presentes[:1]).sum())
    print(f"Estudiantes repetidos según '{id_presentes[0]}': {n_dup_id}")

# 4.3 Valores fuera de la escala de notas
cols_nota = [c for c in df.columns if c.lower().startswith("nota") and pd.api.types.is_numeric_dtype(df[c])]
fuera = {}
for c in cols_nota:
    m = (df[c] < C["SCALE_MIN"]) | (df[c] > C["SCALE_MAX"])
    if m.sum():
        fuera[c] = int(m.sum())
print(f"\nNotas fuera de [{C['SCALE_MIN']}, {C['SCALE_MAX']}]:", fuera if fuera else "ninguna")

# 4.4 Inconsistencias en categorías (mayúsculas, espacios)
for c in df.select_dtypes(include="object").columns:
    if c in C["ID_COLS"] + C["NAME_COLS"] + C["OTHER_SENSITIVE_COLS"]:
        continue
    orig = df[c].nunique()
    norm = df[c].astype(str).str.strip().str.lower().nunique()
    if norm < orig:
        print(f"  '{c}': {orig} categorías, {norm} tras normalizar (hay variantes de escritura)")

## 5. Limpieza y anonimización
Reglas aplicadas (déjalas documentadas en el informe):
- Se eliminan nombres, apellidos y otros datos sensibles.
- DNI y código de estudiante se reemplazan por un **seudónimo irreversible** (hash con sal aleatoria que no se guarda).
- Se eliminan duplicados exactos.
- Notas fuera de escala se marcan como faltantes (probable error de digitación).
- Faltantes: se **documentan** aquí y se imputan **después** de la partición train/test, para no filtrar información del test (ver sección 9).

In [ ]:
def limpiar_y_anonimizar(d, cfg):
    d = d.copy()
    log = []

    # Normalizar texto en categóricas (no en identificadores)
    sensibles = cfg["ID_COLS"] + cfg["NAME_COLS"] + cfg["OTHER_SENSITIVE_COLS"]
    for c in d.select_dtypes(include="object").columns:
        if c not in sensibles:
            d[c] = d[c].astype("string").str.strip()

    # Duplicados
    n0 = len(d)
    d = d.drop_duplicates().reset_index(drop=True)
    log.append(f"Duplicados exactos eliminados: {n0 - len(d)}")

    # Notas fuera de escala -> NaN
    for c in [x for x in d.columns if x.lower().startswith("nota") and pd.api.types.is_numeric_dtype(d[x])]:
        m = (d[c] < cfg["SCALE_MIN"]) | (d[c] > cfg["SCALE_MAX"])
        if m.sum():
            d.loc[m, c] = np.nan
            log.append(f"'{c}': {int(m.sum())} valores fuera de escala convertidos a faltante")

    # Seudonimización (la sal vive solo en memoria: el proceso no es reversible)
    sal = os.urandom(16).hex()
    for c in cfg["ID_COLS"]:
        if c in d.columns:
            d[c] = d[c].astype(str).map(lambda v: hashlib.sha256((sal + v).encode()).hexdigest()[:12])
            log.append(f"'{c}' reemplazado por seudónimo")
    if cfg["ID_COLS"] and cfg["ID_COLS"][0] in d.columns:
        d = d.rename(columns={cfg["ID_COLS"][0]: "id_anonimo"})

    # Eliminar nombres y sensibles
    quitar = [c for c in cfg["NAME_COLS"] + cfg["OTHER_SENSITIVE_COLS"] + cfg["DROP_COLS"] if c in d.columns]
    quitar += [c for c in cfg["ID_COLS"][1:] if c in d.columns]
    d = d.drop(columns=quitar)
    log.append(f"Columnas eliminadas: {quitar}")
    return d, log


df_clean, log_limpieza = limpiar_y_anonimizar(df, C)
for linea in log_limpieza:
    print("•", linea)
print(f"\nRegistros tras limpieza: {len(df_clean):,}")
df_clean.head()

## 6. Variables objetivo y control de fuga de información
- **Clasificación:** `riesgo_academico` = 1 si la nota final es menor al umbral.
- **Regresión:** nota final numérica.

⚠️ **Fuga de información:** la nota final y el riesgo se derivan del mismo dato. Ninguna columna calculada a partir de la nota final (promedio anual, condición final, etc.) puede ser variable de entrada. Define bien **en qué momento** se hace la predicción (por ejemplo, tras el bimestre 2); las notas posteriores a ese momento tampoco deben entrar.

In [ ]:
if C["TARGET_REG"] not in df_clean.columns:
    raise KeyError(f"No existe la columna objetivo '{C['TARGET_REG']}'. Corrige CONFIG['TARGET_REG'].")

df_clean = df_clean.dropna(subset=[C["TARGET_REG"]]).reset_index(drop=True)
df_clean[C["TARGET_CLF"]] = (df_clean[C["TARGET_REG"]] < C["PASS_THRESHOLD"]).astype(int)

vc = df_clean[C["TARGET_CLF"]].value_counts().sort_index()
print("Distribución de la variable objetivo de clasificación:")
print(f"  No en riesgo (0): {vc.get(0, 0):,} ({vc.get(0, 0) / len(df_clean):.1%})")
print(f"  En riesgo   (1): {vc.get(1, 0):,} ({vc.get(1, 0) / len(df_clean):.1%})")

# Detección automática de posible fuga: correlación casi perfecta con la nota final
num = df_clean.select_dtypes(include="number")
corr_t = num.corr()[C["TARGET_REG"]].drop([C["TARGET_REG"], C["TARGET_CLF"]], errors="ignore")
sospechosas = corr_t[corr_t.abs() > 0.95]
print("\nPosible fuga (|correlación| > 0.95 con la nota final):",
      list(sospechosas.index) if len(sospechosas) else "ninguna")

# Se retiran las columnas de fuga declaradas en CONFIG
fuga = [c for c in C["LEAKAGE_COLS"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=fuga)
print("Columnas de fuga retiradas:", fuga if fuga else "ninguna")

## 7. Análisis exploratorio (EDA)
*Alimenta: «Descripción detallada de la interpretación de los resultados (imágenes)».*

In [ ]:
objetivos = [C["TARGET_REG"], C["TARGET_CLF"]]
cols_num = [c for c in df_clean.select_dtypes(include="number").columns if c not in objetivos]
cols_cat = [c for c in df_clean.columns if c not in cols_num + objetivos + ["id_anonimo"]]

print("Estadísticos descriptivos (numéricas):")
display(df_clean[cols_num + [C["TARGET_REG"]]].describe().T.round(2))
print("\nCategóricas:")
for c in cols_cat:
    print(f"  {c}: {df_clean[c].nunique()} categorías -> {df_clean[c].value_counts(dropna=False).head(5).to_dict()}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_clean[C["TARGET_REG"]], bins=20, kde=True, ax=ax[0], color="#3b6fb6")
ax[0].axvline(C["PASS_THRESHOLD"], color="red", ls="--", label=f"Umbral ({C['PASS_THRESHOLD']})")
ax[0].set_title("Distribución de la nota final"); ax[0].legend()
conteo = df_clean[C["TARGET_CLF"]].map({0: "No en riesgo", 1: "En riesgo"}).value_counts()
sns.barplot(x=conteo.index, y=conteo.values, ax=ax[1], palette=["#4c9f70", "#c8553d"])
ax[1].set_title("Balance de clases (riesgo académico)")
for i, v in enumerate(conteo.values):
    ax[1].text(i, v, f"{v} ({v / conteo.sum():.0%})", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(f"{C['OUTPUT_DIR']}/fig_01_objetivo.png", dpi=150)
plt.show()

In [ ]:
n = len(cols_num)
if n:
    cols = 3
    filas = int(np.ceil(n / cols))
    fig, axes = plt.subplots(filas, cols, figsize=(5 * cols, 3.4 * filas))
    for ax, c in zip(np.array(axes).ravel(), cols_num):
        sns.histplot(df_clean[c].dropna(), bins=20, ax=ax, color="#3b6fb6")
        ax.set_title(c)
    for ax in np.array(axes).ravel()[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"{C['OUTPUT_DIR']}/fig_02_distribuciones.png", dpi=150)
    plt.show()

In [ ]:
if cols_num:
    corr = df_clean[cols_num + [C["TARGET_REG"]]].corr()
    plt.figure(figsize=(0.9 * len(corr) + 3, 0.8 * len(corr) + 2))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
    plt.title("Matriz de correlación (Pearson)")
    plt.tight_layout()
    plt.savefig(f"{C['OUTPUT_DIR']}/fig_03_correlacion.png", dpi=150)
    plt.show()

    print("Correlación de cada variable con la nota final:")
    display(corr[C["TARGET_REG"]].drop(C["TARGET_REG"]).sort_values(key=abs, ascending=False).round(3).to_frame("correlación"))

In [ ]:
# Tasa de riesgo y nota promedio por cada variable categórica (con pocas categorías)
cats_plot = [c for c in cols_cat if 2 <= df_clean[c].nunique() <= 12]
if cats_plot:
    fig, axes = plt.subplots(len(cats_plot), 2, figsize=(12, 3.2 * len(cats_plot)), squeeze=False)
    for i, c in enumerate(cats_plot):
        sns.barplot(data=df_clean, x=c, y=C["TARGET_CLF"], ax=axes[i, 0], color="#c8553d", errorbar=None)
        axes[i, 0].set_title(f"Proporción en riesgo por {c}")
        sns.boxplot(data=df_clean, x=c, y=C["TARGET_REG"], ax=axes[i, 1], color="#9bb7dc")
        axes[i, 1].set_title(f"Nota final por {c}")
    plt.tight_layout()
    plt.savefig(f"{C['OUTPUT_DIR']}/fig_04_categoricas.png", dpi=150)
    plt.show()

In [ ]:
# Comparación de variables numéricas entre estudiantes en riesgo y no en riesgo
if cols_num:
    comp = df_clean.groupby(C["TARGET_CLF"])[cols_num].mean().T
    comp.columns = ["No en riesgo", "En riesgo"]
    comp["diferencia"] = comp["En riesgo"] - comp["No en riesgo"]
    display(comp.round(2))

## 8. Resumen automático de insights
Texto de apoyo para redactar la interpretación. **Revísalo y reescríbelo con tus palabras** antes de pasarlo al Word.

In [ ]:
ins = []
ins.append(f"El dataset final tiene {len(df_clean):,} registros y {df_clean.shape[1]} columnas.")
ins.append(f"{df_clean[C['TARGET_CLF']].mean():.1%} de los estudiantes está en riesgo académico (nota final < {C['PASS_THRESHOLD']}).")
if cols_num:
    top = corr[C["TARGET_REG"]].drop(C["TARGET_REG"]).abs().sort_values(ascending=False).head(3)
    for k in top.index:
        ins.append(f"'{k}' se asocia con la nota final (r = {corr.loc[k, C['TARGET_REG']]:+.2f}).")
ratio = df_clean[C["TARGET_CLF"]].value_counts(normalize=True).min()
if ratio < 0.2:
    ins.append("Las clases están desbalanceadas: conviene usar partición estratificada y métricas como F1, recall y AUC, no solo accuracy.")
for t in ins:
    print("•", t)

## 9. Partición train/test fija y exportación
Esta partición la deben usar **todos** los compañeros para que el benchmark compare modelos en igualdad de condiciones (misma semilla, mismos registros).

La imputación de faltantes **no** se hace aquí: cada pipeline debe imputar con estadísticos calculados solo con el conjunto de entrenamiento (por ejemplo `SimpleImputer` dentro de un `Pipeline`).

In [ ]:
y_clf, y_reg = df_clean[C["TARGET_CLF"]], df_clean[C["TARGET_REG"]]

estratificar = y_clf if y_clf.value_counts().min() >= 2 else None
train, test = train_test_split(
    df_clean, test_size=C["TEST_SIZE"], random_state=C["SEED"], stratify=estratificar
)
print(f"Train: {len(train):,} ({len(train) / len(df_clean):.0%}) | Test: {len(test):,} ({len(test) / len(df_clean):.0%})")
print("Proporción en riesgo -> train: {:.1%} | test: {:.1%}".format(train[C["TARGET_CLF"]].mean(), test[C["TARGET_CLF"]].mean()))

out = C["OUTPUT_DIR"]
df_clean.to_csv(f"{out}/dataset_limpio_anonimizado.csv", index=False)
train.to_csv(f"{out}/train.csv", index=False)
test.to_csv(f"{out}/test.csv", index=False)

dic_final = diccionario(df_clean)
dic_final.loc[dic_final["columna"] == C["TARGET_CLF"], "descripcion"] = f"1 = en riesgo (nota final < {C['PASS_THRESHOLD']}), 0 = no. Objetivo de CLASIFICACIÓN"
dic_final.loc[dic_final["columna"] == C["TARGET_REG"], "descripcion"] = "Objetivo de REGRESIÓN"
dic_final.loc[dic_final["columna"] == "id_anonimo", "descripcion"] = "Identificador seudonimizado (no reversible)"
dic_final.to_csv(f"{out}/diccionario_datos.csv", index=False, encoding="utf-8-sig")
dic_final.to_excel(f"{out}/diccionario_datos.xlsx", index=False)

print("\nArchivos generados en", out + "/")
for f in sorted(os.listdir(out)):
    print("  -", f)
if ES_DEMO:
    print("\nRECUERDA: esta ejecución usó datos sintéticos de demostración.")
dic_final

## 10. Checklist antes de entregar a tus compañeros
- [ ] Ejecuté con la data real (no aparece el aviso de datos sintéticos).
- [ ] Completé `DESCRIPCIONES` con el significado real de cada columna.
- [ ] Verifiqué que no queda nombre, DNI ni dato identificable en `dataset_limpio_anonimizado.csv`.
- [ ] Definí con el equipo en qué momento se predice y retiré las columnas de fuga (`LEAKAGE_COLS`).
- [ ] Confirmé el umbral de aprobación y la escala de notas de la IE (en primaria puede ser AD/A/B/C y requiere convertirse).
- [ ] Compartí `train.csv`, `test.csv` y `diccionario_datos.xlsx`.